In [1]:
# --------------------------------------------------
# Import required libraries for model training,
# evaluation, and saving trained models
# --------------------------------------------------
import pandas as pd
import os
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import make_scorer, f1_score
import pandas as pd



In [2]:
# --------------------------------------------------
# Load preprocessed training and testing datasets
# generated from the preprocessing pipeline
# --------------------------------------------------
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test  = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze() # .squeeze() to convert DataFrame to Series
y_test  = pd.read_csv("../data/processed/y_test.csv").squeeze() # .squeeze() to convert DataFrame to Series

X_train.shape, X_test.shape, y_train.shape, y_test.shape


((316, 41), (79, 41), (316,), (79,))

In [3]:
# --------------------------------------------------
# Initialize all models used in the project
# --------------------------------------------------
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    
    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    ),
    
    "Support Vector Machine": SVC(
        kernel="rbf",
        probability=True,   # Required for SHAP & LIME
        random_state=42
    )
}


In [4]:
# --------------------------------------------------
# Train each model and evaluate using accuracy
# --------------------------------------------------
trained_models = {}
results = []

for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    
    trained_models[model_name] = model
    results.append({
        "Model": model_name,
        "Accuracy": accuracy
    })
    
    print(f"{model_name} trained successfully | Accuracy: {accuracy:.4f}")


Logistic Regression trained successfully | Accuracy: 0.8861
Decision Tree trained successfully | Accuracy: 0.8608
Random Forest trained successfully | Accuracy: 0.8861
Gradient Boosting trained successfully | Accuracy: 0.8861
Support Vector Machine trained successfully | Accuracy: 0.8861


In [5]:
# --------------------------------------------------
# Save all trained models to the models directory
# --------------------------------------------------
os.makedirs("../models", exist_ok=True)

for model_name, model in trained_models.items():
    filename = model_name.lower().replace(" ", "_") + ".pkl"
    joblib.dump(model, f"../models/{filename}")

print("All trained models saved successfully.")


All trained models saved successfully.


In [6]:
# --------------------------------------------------
# Display accuracy comparison of all models
# --------------------------------------------------
results_df = pd.DataFrame(results)
results_df.sort_values(by="Accuracy", ascending=False)


,Model,Accuracy
0,Logistic Regression,0.886076
2,Random Forest,0.886076
3,Gradient Boosting,0.886076
4,Support Vector Machine,0.886076
1,Decision Tree,0.860759


In [7]:
# -------------------------------------------------------
# 5-Fold Stratified Cross-Validation
# -------------------------------------------------------
scoring = {
    'accuracy':  'accuracy',
    'f1':        make_scorer(f1_score),
    'precision': 'precision',
    'recall':    'recall',
    'roc_auc':   'roc_auc',
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []

for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    cv_results.append({
        'Model':         name,
        'Accuracy':      f"{scores['test_accuracy'].mean():.4f} ± {scores['test_accuracy'].std():.4f}",
        'F1-Score':      f"{scores['test_f1'].mean():.4f} ± {scores['test_f1'].std():.4f}",
        'Precision':     f"{scores['test_precision'].mean():.4f} ± {scores['test_precision'].std():.4f}",
        'Recall':        f"{scores['test_recall'].mean():.4f} ± {scores['test_recall'].std():.4f}",
        'ROC-AUC':       f"{scores['test_roc_auc'].mean():.4f} ± {scores['test_roc_auc'].std():.4f}",
    })

cv_df = pd.DataFrame(cv_results).sort_values('F1-Score', ascending=False)
cv_df

,Model,Accuracy,F1-Score,Precision,Recall,ROC-AUC
3,Gradient Boosting,0.9335 ± 0.0324,0.9507 ± 0.0244,0.9443 ± 0.0240,0.9575 ± 0.0278,0.9812 ± 0.0127
2,Random Forest,0.9272 ± 0.0192,0.9463 ± 0.0144,0.9361 ± 0.0219,0.9574 ± 0.0235,0.9775 ± 0.0129
0,Logistic Regression,0.9114 ± 0.0357,0.9339 ± 0.0274,0.9300 ± 0.0206,0.9385 ± 0.0417,0.9741 ± 0.0164
1,Decision Tree,0.9050 ± 0.0364,0.9286 ± 0.0299,0.9252 ± 0.0168,0.9337 ± 0.0532,0.8902 ± 0.0301
4,Support Vector Machine,0.8543 ± 0.0538,0.8941 ± 0.0422,0.8636 ± 0.0274,0.9288 ± 0.0692,0.9452 ± 0.0271


In [8]:
cv_df.to_csv("../evaluation/cv_results.csv", index=False)
print("Cross-validation results saved.")

Cross-validation results saved.


In [9]:
print("MODEL TRAINING COMPLETED")
print("-" * 50)
print(f"Total models trained: {len(trained_models)}")
print("Models are ready for evaluation and explainability analysis.")


MODEL TRAINING COMPLETED
--------------------------------------------------
Total models trained: 5
Models are ready for evaluation and explainability analysis.
